# Lab type: review
# Course: ML201 — Applied Machine Learning
# Lesson: Gradient Boosting: XGBoost, LightGBM, and CatBoost
# Task: The code below is correct and working. Read each section, run it, then answer the judgment questions in the markdown cells below each block.

In [ ]:
!pip install xgboost lightgbm catboost --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
import lightgbm as lgb
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

np.random.seed(42)
n = 3000

tenure_months = np.random.exponential(30, n).clip(1, 120).astype(int)
performance_score = np.random.uniform(1, 5, n).round(1)
salary = np.random.normal(75000, 25000, n).clip(30000, 200000)
num_projects = np.random.randint(1, 10, n)
dept_raw = np.random.choice(['engineering', 'sales', 'marketing', 'hr'], n,
                             p=[0.35, 0.30, 0.20, 0.15])
contract_type_raw = np.random.choice(['permanent', 'contract'], n, p=[0.7, 0.3])

log_odds = (
    -2.5
    - 0.02 * tenure_months
    - 0.3 * performance_score
    - 0.000005 * salary
    + 0.1 * num_projects
    + (contract_type_raw == 'contract').astype(float) * 0.8
)
prob_churn = 1 / (1 + np.exp(-log_odds))
churned = (np.random.rand(n) < prob_churn).astype(int)

# Encode categoricals for XGBoost / LightGBM
le_dept = LabelEncoder()
le_contract = LabelEncoder()
dept_enc = le_dept.fit_transform(dept_raw)
contract_enc = le_contract.fit_transform(contract_type_raw)

df_enc = pd.DataFrame({
    'tenure_months': tenure_months,
    'performance_score': performance_score,
    'salary': salary,
    'num_projects': num_projects,
    'dept': dept_enc,
    'contract_type': contract_enc
})
y = pd.Series(churned, name='churned')

# Raw string categoricals for CatBoost
df_cat = pd.DataFrame({
    'tenure_months': tenure_months,
    'performance_score': performance_score,
    'salary': salary,
    'num_projects': num_projects,
    'dept': dept_raw,
    'contract_type': contract_type_raw
})

# Encoded splits (XGBoost / LightGBM)
X_train_full, X_test, y_train_full, y_test = train_test_split(
    df_enc, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full
)

# Categorical splits (CatBoost)
X_train_full_cat, X_test_cat, y_train_full_cat, y_test_cat = train_test_split(
    df_cat, y, test_size=0.2, random_state=42, stratify=y
)
X_train_cat, X_val_cat, y_train_cat, y_val_cat = train_test_split(
    X_train_full_cat, y_train_full_cat, test_size=0.2, random_state=42, stratify=y_train_full_cat
)

print('Split sizes (encoded):')
print(f'  X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}')
print(f'Class balance (full dataset):\n{y.value_counts(normalize=True).round(3)}')

## Part 1: XGBoost with Early Stopping

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='auc',
    early_stopping_rounds=30,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

xgb_best_iter = xgb_model.best_iteration
xgb_test_auc = roc_auc_score(y_test, xgb_model.predict_proba(X_test)[:, 1])

print(f'XGBoost best_iteration : {xgb_best_iter}')
print(f'XGBoost test AUC       : {xgb_test_auc:.4f}')

**Question 1:** The `eval_set` uses `X_val` (carved from `X_train_full`), not `X_test`. What goes wrong if you pass `eval_set=[(X_test, y_test)]` for early stopping? Why does this make the final test AUC optimistic?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q1</summary>

**What goes wrong:** Using `eval_set=[(X_test, y_test)]` makes the test set part of the model selection process — the number of boosting rounds is chosen specifically to maximise performance on that test set. When you later evaluate on the same test set, the reported AUC is optimistic because you have implicitly tuned the model to it.

**Why the AUC is optimistic:** Early stopping is a form of hyperparameter selection (selecting the optimal `n_estimators`). Any time you use a dataset both to make a model selection decision and to estimate its performance, the estimate is upwardly biased — the data "told" the training when to stop, so the model is slightly tailored to it.

**Correct approach:** Carve `X_val` from `X_train_full` (as done in this notebook), use it for early stopping, and keep `X_test` unseen until final evaluation.

</details>

**Question 2:** `early_stopping_rounds=30` stops training when validation AUC has not improved for 30 consecutive rounds. If you set this too low (e.g., 5 rounds), what failure mode occurs? If too high (e.g., 500 rounds)?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q2</summary>

**Too low (e.g., 5 rounds):** Training stops prematurely when the first brief plateau appears in validation AUC, even if it is just noise. The model is undertrained — it stops before it has found the best solution, leaving predictive performance on the table. This is equivalent to stopping at a local minimum in the validation curve.

**Too high (e.g., 500 rounds):** The window is so generous that the model continues adding trees long past the true validation optimum. Later rounds fit noise in the training set and degrade generalisation. You also waste compute time and memory because hundreds of trees are added after the real performance peak has passed.

**Rule of thumb:** Set `early_stopping_rounds` to roughly 10–20% of your expected `n_estimators`, or tune it with a small holdout experiment. Start around 30–50 for most problems.

</details>

## Part 2: LightGBM — Leaf-wise Growth

In [ ]:
lgbm_model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
lgbm_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)]
)

lgbm_best_iter = lgbm_model.best_iteration_
lgbm_test_auc = roc_auc_score(y_test, lgbm_model.predict_proba(X_test)[:, 1])

print(f'LightGBM best_iteration_: {lgbm_best_iter}')
print(f'LightGBM test AUC       : {lgbm_test_auc:.4f}')

**Question 3:** LightGBM uses leaf-wise tree growth while XGBoost uses level-wise. What does this mean in practice for small datasets (fewer than 500 rows)? Which is more likely to overfit and why?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q3</summary>

**Leaf-wise on small datasets:** LightGBM always splits the single leaf with the highest current loss reduction, regardless of tree depth. This quickly creates a very deep, asymmetric branch that fits a handful of examples very precisely — on a small dataset (< 500 rows), this almost always overfits because individual examples carry disproportionate weight in the leaf's loss reduction calculation.

**XGBoost is less prone to overfitting on small data** because level-wise growth expands all leaves at the same depth simultaneously. Adding a new level requires the same split quality across all existing leaves — it cannot drill down on a single promising branch unless the signal is consistent across the entire tree level.

**Mitigation for LightGBM:** Use `min_data_in_leaf` (default 20) to require a minimum number of examples per leaf, which limits how deeply the tree can fit individual examples.

</details>

**Question 4:** `num_leaves=31` in LightGBM roughly corresponds to a `max_depth` of 5 in a balanced binary tree (2^5 = 32 leaves). If you increase `num_leaves` to 127 but keep `learning_rate` the same, what happens to the bias-variance tradeoff? What complementary hyperparameter should you also adjust?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q4</summary>

**Bias-variance tradeoff:** Increasing `num_leaves` to 127 dramatically increases model capacity — each tree can now represent a much more complex function, reducing bias. But with the same `learning_rate`, each complex tree's contribution is too strong, driving the ensemble toward overfitting. Variance increases sharply.

**Complementary hyperparameter to adjust:** Reduce `learning_rate` proportionally (e.g., from 0.05 to 0.01–0.02) when increasing `num_leaves`. More leaves means each tree is more complex and contributes more signal per iteration; a smaller learning_rate shrinks each tree's contribution, requiring more iterations to converge but regularising the model against overfitting. Also consider increasing `min_data_in_leaf` to prevent tiny leaf splits, and enabling `reg_lambda`/`reg_alpha` (L2/L1 regularisation).

</details>

## Part 3: CatBoost — Native Categorical Support

In [ ]:
catboost_model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.05,
    depth=5,
    random_seed=42,
    verbose=0
)
catboost_model.fit(
    X_train_cat, y_train_cat,
    cat_features=['dept', 'contract_type'],
    eval_set=(X_val_cat, y_val_cat),
    early_stopping_rounds=30
)

catboost_best_iter = catboost_model.best_iteration_
catboost_test_auc = roc_auc_score(y_test_cat, catboost_model.predict_proba(X_test_cat)[:, 1])

print(f'CatBoost best_iteration_: {catboost_best_iter}')
print(f'CatBoost test AUC       : {catboost_test_auc:.4f}')

**Question 5:** CatBoost handles `dept` and `contract_type` without one-hot or label encoding. What is the risk of using naive mean target encoding for categorical features outside a proper cross-validation pipeline? This is the problem CatBoost's ordered target statistics are designed to solve — describe it in concrete terms.

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q5</summary>

**The risk of naive mean target encoding:** Computing "average target per category" globally on the training set causes each training example's encoding to include information from its own label. For a category with only one example, the encoding equals that example's label exactly — the model sees the answer as a feature, creating a severe information leak that inflates training performance. For rare categories, this effect is extreme.

**The concrete problem:** Consider a product SKU seen only once in training data, and that transaction was fraudulent (label=1). Naive encoding gives it a value of 1.0. At training time the model "sees" the fraud label disguised as a feature — it trivially learns to predict fraud for that SKU. But at inference time on a new transaction with the same SKU, the encoding is still 1.0 even if this is a legitimate transaction, causing the model to over-flag it.

**CatBoost's solution:** Ordered target statistics compute the target mean for each category using only the examples that appeared *before* the current example in a randomly permuted order — never including the current example itself — preventing each training example from seeing its own label.

</details>

**Question 6:** You have a dataset with five high-cardinality categorical columns (`city`, `product_sku`, `user_segment`, etc.), each with 200+ unique values. Of XGBoost, LightGBM, and CatBoost, which would you choose and why?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q6</summary>

**Choose CatBoost.** With 200+ unique values per column, one-hot encoding creates 1000+ sparse binary columns — tree split evaluation on thousands of binary columns is slow and memory-intensive for both XGBoost and LightGBM, and OHE loses the structural relationship between categories. Label encoding introduces arbitrary ordinal relationships that mislead the model.

CatBoost's native ordered target statistics handle high cardinality robustly without encoding preprocessing, avoiding leakage and the cardinality explosion. LightGBM does offer native categorical support (`cat_features`), but it uses a different algorithm that can still struggle at 200+ unique values. CatBoost was specifically designed for this regime and typically outperforms manual encoding alternatives.

</details>

## Part 4: Comparison

In [ ]:
results = pd.DataFrame([
    {'Library': 'XGBoost',   'Best Iteration': xgb_best_iter,      'Test AUC': xgb_test_auc},
    {'Library': 'LightGBM',  'Best Iteration': lgbm_best_iter,     'Test AUC': lgbm_test_auc},
    {'Library': 'CatBoost',  'Best Iteration': catboost_best_iter, 'Test AUC': catboost_test_auc},
])

print('\n=== Model Comparison ===')
print(results.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#50fa7b', '#8be9fd', '#ff79c6']
bars = ax.bar(results['Library'], results['Test AUC'], color=colors, edgecolor='white')
ax.set_ylim(0.5, 1.0)
ax.set_ylabel('Test AUC')
ax.set_title('Test AUC by Gradient Boosting Library')
for bar, val in zip(bars, results['Test AUC']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
            f'{val:.4f}', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

**Question 7:** All three libraries achieved similar AUC on this dataset. What practical factors would still lead you to choose one over the others in a real production project? Name at least two factors and explain the tradeoff each one represents.

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q7</summary>

**Factor 1 — Training speed and infrastructure cost:** LightGBM is typically the fastest of the three (histogram-based with leaf-wise growth) and should be preferred when training time, compute budget, or fast iteration matters. The tradeoff: LightGBM requires careful hyperparameter tuning (`num_leaves`, `min_data_in_leaf`) to avoid overfitting, especially on small datasets.

**Factor 2 — Data preprocessing burden vs team capacity:** CatBoost eliminates all categorical preprocessing (no OHE, no label encoding, no target encoding plumbing), which reduces engineering effort and bug surface in production pipelines. The tradeoff: CatBoost has slower training than LightGBM and a less mature sklearn-compatible API, requiring teams to invest in learning its inference library for deployment.

Other valid factors: deployment environment constraints (ONNX export, language bindings), community support and documentation quality, or regulatory requirements (some industries require model explainability tooling that integrates better with XGBoost/LightGBM).

</details>

## Summary

Check your understanding with these final questions. Each should be answerable in one sentence.

1. Why does using the held-out test set as the early stopping validation set produce an optimistic test AUC, and what is the correct remedy?

2. In one sentence, explain why LightGBM's leaf-wise growth makes it more prone to overfitting on small datasets than XGBoost's level-wise growth.

3. You have a production dataset where new categories appear in live scoring that were not seen during training. Which library's native categorical handling degrades most gracefully in this situation, and why?

<details>
<summary>🔑 Reveal summary answers</summary>

1. **Test set as early stopping validation:** Using the test set for early stopping selects the number of rounds that maximises that specific test set's AUC — the optimisation target and the evaluation target are the same data, producing an optimistic estimate; the remedy is to hold out a separate validation set from training data for early stopping and keep the test set completely unseen.

2. **LightGBM leaf-wise overfitting on small data:** Leaf-wise growth always splits the single highest-loss leaf, potentially creating a very deep branch that fits a handful of examples precisely; on small datasets individual examples heavily influence leaf splits, while XGBoost's level-wise growth distributes splits evenly across all current leaves and requires consistent signal at each depth level.

3. **New categories at inference time:** CatBoost degrades most gracefully because its ordered target statistics naturally fall back toward the global target mean for unseen categories; XGBoost with label encoding produces an out-of-range integer that the model has never seen, and LightGBM's native categorical handling also uses a mean-based fallback, but CatBoost's ordered approach is more robust because it was designed from the ground up for this scenario.

</details>